# Task 9: классификация твитов (positive/negative)

Ноутбук повторяет подход из референса: TF-IDF по словам и символам + `LinearSVC`, выбор лучшей модели по валидации и формирование предсказаний для теста в формате Python-списка.

In [1]:
%pip install -q scikit-learn scipy numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import io
import zipfile
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

TRAIN_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Social_Networks/Task_2/train.csv.zip"
VAL_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Social_Networks/Task_2/val.csv.zip"


def load_csv(source: str) -> pd.DataFrame:
    is_url = source.startswith("http://") or source.startswith("https://")
    is_zip = source.lower().endswith(".zip")

    if not is_zip:
        return pd.read_csv(source)

    if is_url:
        with urlopen(source) as response:
            zip_bytes = response.read()
        zf = zipfile.ZipFile(io.BytesIO(zip_bytes))
    else:
        zf = zipfile.ZipFile(source)

    with zf:
        csv_candidates = [
            name
            for name in zf.namelist()
            if name.lower().endswith(".csv") and not Path(name).name.startswith("._")
        ]
        if not csv_candidates:
            raise ValueError(f"No CSV files found in zip: {source}")

        # Берем первый нормальный CSV (игнорируя __MACOSX/._*).
        csv_name = csv_candidates[0]
        with zf.open(csv_name) as csv_file:
            return pd.read_csv(csv_file)

In [3]:
train_df = load_csv(TRAIN_URL)
val_df = load_csv(VAL_URL)

print("train:", train_df.shape)
print("val:", val_df.shape)
train_df.head(2)

train: (181467, 3)
val: (22683, 3)


,id,text,class
0,0,VitaliaPerilova у тебя классные очкиони тебя о...,1
1,1,Никогда не подумала бы что самое приятное на с...,1


In [4]:
train_df["text"] = train_df["text"].fillna("").astype(str)
val_df["text"] = val_df["text"].fillna("").astype(str)

X_train = train_df["text"]
y_train = train_df["class"].astype(int)

X_val = val_df["text"]
y_val = val_df["class"].astype(int)

In [5]:
def make_model(word_ngram, char_ngram, c_value, max_df):
    return Pipeline(
        [
            (
                "features",
                FeatureUnion(
                    [
                        (
                            "word",
                            TfidfVectorizer(
                                analyzer="word",
                                ngram_range=word_ngram,
                                min_df=2,
                                max_df=max_df,
                                sublinear_tf=True,
                            ),
                        ),
                        (
                            "char",
                            TfidfVectorizer(
                                analyzer="char_wb",
                                ngram_range=char_ngram,
                                min_df=2,
                                sublinear_tf=True,
                            ),
                        ),
                    ]
                ),
            ),
            ("clf", LinearSVC(C=c_value)),
        ]
    )


candidates = [
    ("model_1", make_model((1, 2), (3, 5), 1.0, 0.98)),
    ("model_2", make_model((1, 3), (2, 6), 0.7, 0.99)),
]

scores = []
for name, model in candidates:
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_pred)
    scores.append((name, model, val_acc))
    print(f"{name}: val_acc = {val_acc:.6f}")

best_name, best_model, best_val_acc = max(scores, key=lambda x: x[2])
print(f"Best model: {best_name}, val_acc = {best_val_acc:.6f}")

model_1: val_acc = 0.794516
model_2: val_acc = 0.799321
Best model: model_2, val_acc = 0.799321


In [6]:
full_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
X_full = full_df["text"]
y_full = full_df["class"].astype(int)

best_model.fit(X_full, y_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('features', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('word', ...), ('char', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'


In [10]:
# Вставьте сюда ссылку на тестовый CSV/ZIP.
# Для быстрой проверки можно временно поставить VAL_URL.
TEST_PATH_OR_URL = '/Users/afafos/Desktop/Program/itmo/automatic-text-processing-itmo/Soc_Net_Task_2_test_9.csv'

if TEST_PATH_OR_URL.strip():
    test_df = load_csv(TEST_PATH_OR_URL)
    test_df["text"] = test_df["text"].fillna("").astype(str)

    test_pred = best_model.predict(test_df["text"])
    predictions = test_pred.astype(int).tolist()

    print("Predictions count:", len(predictions))
    print("Первые 20:", predictions[:20])
    print(predictions)

    with open("task-9-predictions.txt", "w", encoding="utf-8") as f:
        f.write(str(predictions))
    print("Сохранено в task-9-predictions.txt")
else:
    print("Укажите TEST_PATH_OR_URL (ссылка или локальный путь к test.csv/test.zip)")

Predictions count: 1000
Первые 20: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 